# Feature Engineering

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer
import joblib
warnings.filterwarnings('ignore')

In [2]:
df_weekly = pd.read_csv('../data/aggregated/weekly_vacancy_counts.csv')
df_weekly

,job_title_unified,job_posted_date,vacancy_count
0,Business Analyst,2023-01-01,182
1,Business Analyst,2023-01-08,1060
2,Business Analyst,2023-01-15,1119
3,Business Analyst,2023-01-22,1156
4,Business Analyst,2023-01-29,1047
...,...,...,...
366,Software Engineer,2023-12-03,841
367,Software Engineer,2023-12-10,1032
368,Software Engineer,2023-12-17,899
369,Software Engineer,2023-12-24,817


In [3]:
def create_features(
    df: pd.DataFrame,
    group_col: str = 'job_title_unified',
    time_col: str = 'job_posted_date'
) -> pd.DataFrame:
    """Creates time series features for the given DataFrame grouped by the specified column."""

    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df.sort_values([group_col, time_col], inplace=True)
    df.reset_index(drop=True, inplace=True)

    grp = df.groupby(group_col, sort=False)['vacancy_count']

    # Date features
    df['month'] = df[time_col].dt.month
    df['quarter'] = df[time_col].dt.quarter
    df['is_month_start'] = df[time_col].dt.is_month_start.astype(int)
    df['week_of_year'] = df[time_col].dt.isocalendar().week.astype(int)
    df['is_month_end'] = df[time_col].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df[time_col].dt.is_quarter_start.astype(int)
    df['is_quarter_end'] = df[time_col].dt.is_quarter_end.astype(int)
    df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

    # Lag features
    df['lag_1'] = grp.shift(1)
    df['lag_2'] = grp.shift(2)
    df['lag_3'] = grp.shift(3)
    df['lag_4'] = grp.shift(4)

    # Rolling mean features inside each group
    df['rolling_mean_2'] = grp.transform(
        lambda s: s.shift(1).rolling(window=2, min_periods=2).mean()
    )
    df['rolling_mean_4'] = grp.transform(
        lambda s: s.shift(1).rolling(window=4, min_periods=4).mean()
    )

    # Residual features based on grouped rolling mean
    residual = df['vacancy_count'] - df['rolling_mean_2']
    df['residual_lag1'] = residual.groupby(df[group_col], sort=False).shift(1)
    df['residual_lag2'] = residual.groupby(df[group_col], sort=False).shift(2)

    # Rolling std features inside each group
    df['rolling_std_2'] = grp.transform(
        lambda s: s.shift(1).rolling(window=2, min_periods=2).std()
    )
    df['rolling_std_4'] = grp.transform(
        lambda s: s.shift(1).rolling(window=4, min_periods=4).std()
    )

    # EWM inside each group
    df['ewm_0.1'] = grp.transform(
        lambda s: s.shift(1).ewm(alpha=0.1, adjust=False).mean()
    )
    df['ewm_0.5'] = grp.transform(
        lambda s: s.shift(1).ewm(alpha=0.5, adjust=False).mean()
    )
    df['ewm_0.9'] = grp.transform(
        lambda s: s.shift(1).ewm(alpha=0.9, adjust=False).mean()
    )

    # Expanding features inside each group
    df['expanding_mean'] = grp.transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    df['expanding_std'] = grp.transform(
        lambda s: s.shift(1).expanding(min_periods=2).std()
    )
    df['expanding_max'] = grp.transform(
        lambda s: s.shift(1).expanding(min_periods=1).max()
    )
    df['expanding_min'] = grp.transform(
        lambda s: s.shift(1).expanding(min_periods=1).min()
    )

    # Difference features inside each group
    df['diff_1'] = grp.transform(lambda s: s.shift(1) - s.shift(2))
    df['diff_2'] = grp.transform(lambda s: s.shift(2) - s.shift(3))
    df['diff_3'] = grp.transform(lambda s: s.shift(3) - s.shift(4))

    # Percentage change features inside each group
    def safe_pct_change(series, lag_a, lag_b):
        a = series.shift(lag_a)
        b = series.shift(lag_b)
        return np.where(b != 0, a / b - 1, np.nan)

    df['pct_change_1'] = grp.transform(lambda s: safe_pct_change(s, 1, 2))
    df['pct_change_2'] = grp.transform(lambda s: safe_pct_change(s, 2, 3))
    df['pct_change_3'] = grp.transform(lambda s: safe_pct_change(s, 3, 4))

    # Calendar features
    m = df[time_col].dt.month
    w = df['week_of_year']
    df['is_main_hiring_peak'] = (m.isin([1, 2])).astype(int)
    df['is_summer_hiring_peak'] = (m == 8).astype(int)
    df['is_may_slowdown'] = (m == 5).astype(int)
    df['is_december_slowdown'] = (m == 12).astype(int)
    df['is_post_peak_spring'] = (m.isin([3, 4])).astype(int)
    df['is_early_summer_recovery'] = (m.isin([6, 7])).astype(int)
    df['is_stable_plateau'] = (m.isin([9, 10, 11])).astype(int)
    df['is_q1_peak'] = (df['quarter'] == 1).astype(int)
    df['is_q2_low'] = (df['quarter'] == 2).astype(int)
    df['is_q3_second'] = (df['quarter'] == 3).astype(int)
    df['is_new_year_holidays'] = ((m == 1) & (w.between(1, 2))).astype(int)
    df['is_may_holidays'] = ((m == 5) & (w.between(18, 19))).astype(int)
    df['is_nauryz_week'] = ((m == 3) & (w.between(11, 12))).astype(int)
    df['is_graduation_season'] = (m.isin([6, 7])).astype(int)

    return df

In [4]:
# Create features and drop rows with NaN values resulting from lag and rolling calculations
df_weekly = create_features(df_weekly)
df_weekly.dropna(inplace=True)
df_weekly.head(10)

,job_title_unified,job_posted_date,vacancy_count,month,quarter,is_month_start,week_of_year,is_month_end,is_quarter_start,is_quarter_end,...,is_post_peak_spring,is_early_summer_recovery,is_stable_plateau,is_q1_peak,is_q2_low,is_q3_second,is_new_year_holidays,is_may_holidays,is_nauryz_week,is_graduation_season
4,Business Analyst,2023-01-29,1047,1,1,0,4,0,0,0,...,0,0,0,1,0,0,0,0,0,0
5,Business Analyst,2023-02-05,982,2,1,0,5,0,0,0,...,0,0,0,1,0,0,0,0,0,0
6,Business Analyst,2023-02-12,880,2,1,0,6,0,0,0,...,0,0,0,1,0,0,0,0,0,0
7,Business Analyst,2023-02-19,869,2,1,0,7,0,0,0,...,0,0,0,1,0,0,0,0,0,0
8,Business Analyst,2023-02-26,766,2,1,0,8,0,0,0,...,0,0,0,1,0,0,0,0,0,0
9,Business Analyst,2023-03-05,911,3,1,0,9,0,0,0,...,1,0,0,1,0,0,0,0,0,0
10,Business Analyst,2023-03-12,806,3,1,0,10,0,0,0,...,1,0,0,1,0,0,0,0,0,0
11,Business Analyst,2023-03-19,682,3,1,0,11,0,0,0,...,1,0,0,1,0,0,0,0,1,0
12,Business Analyst,2023-03-26,661,3,1,0,12,0,0,0,...,1,0,0,1,0,0,0,0,1,0
13,Business Analyst,2023-04-02,785,4,2,0,13,0,0,0,...,1,0,0,0,1,0,0,0,0,0


In [5]:
# Split the data into training and testing sets based on the time column
def split_time_series(df: pd.DataFrame, group_col: str = 'job_title_unified', time_col: str = 'job_posted_date', test_size: int = 8) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Splits the DataFrame into training and testing sets based on the specified time column.'''
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df.sort_values([group_col, time_col], inplace=True)
    split_point = df[time_col].iloc[-test_size]
    train_df = df[df[time_col] < split_point]
    test_df = df[df[time_col] >= split_point]
    return train_df, test_df

In [6]:
# Perform the split and print the shapes of the resulting sets
train,test = split_time_series(df_weekly)
print(f'Training set shape: {train.shape}')
print(f'Testing set shape: {test.shape}')

Training set shape: (287, 49)
Testing set shape: (56, 49)


In [7]:
# Prepare the feature matrices and target vectors for modeling
X_train = train.drop(columns=['vacancy_count', 'job_posted_date'])
y_train = train['vacancy_count']
X_test = test.drop(columns=['vacancy_count', 'job_posted_date'])
y_test = test['vacancy_count']

In [8]:
# Identify numeric and categorical features for preprocessing
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
# Define pipelines for numeric and categorical features
num_pipe = Pipeline([
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('encoder', TargetEncoder(cv=5,smooth='auto',random_state=42,target_type='continuous'))
])
# Combine pipelines into a ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features)
], remainder='drop',verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [9]:
# Fit the preprocessor on the training data and transform both training and testing sets
preprocessor.fit(X_train,y_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [10]:
joblib.dump(preprocessor, '../models/preprocessor_demand.joblib')

['../models/preprocessor_demand.joblib']